# WRAP_P6_CASCADE_LAB — Notebook 07

**Spec ref:** §9.5 — Cascade detection + forecasting.

Answers: *when should we abstain or flip to risk-off?*

## Section 00 — Env & Imports

In [1]:
import sys, logging, pickle
from pathlib import Path
import numpy as np
import pandas as pd

_NB_DIR = Path('<repo>/p6lab/notebooks')
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))
import _common
from _common import (
    NOTEBOOK_DATA_SLICE, collect_snapshots, versioned_dir,
)
_DS       = NOTEBOOK_DATA_SLICE
SYMBOL    = _DS['symbol']
TICK_SIZE = _DS['tick_size']

_P6LAB        = Path(_common._P6LAB_ROOT)
ARTIFACTS_DIR = _P6LAB / 'artifacts' / 'p6lab'

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger(__name__)

from p6lab.features._l1_adapter import L1Adapter, L1AdapterConfig
from p6lab.features.l1_features import L1FeatureNames, compute_l1_features
from p6lab.features.l2_features import (
    L2FeatureNames, L2History, L2Snapshot, compute_l2_features,
)
from p6lab.features.fragility_index import FragilityIndex
from p6lab.validation.augmentation import AugmentationEngine
from p6lab.validation.cpcv import CascadeAwareCPCV
from p6lab.validation.information_gain import must_beat_baseline
from p6lab.patterns.cascade_taxonomy import (
    CascadeClassifier, CascadeType, cascade_events_to_df,
)
from sklearn.metrics import roc_auc_score

np.random.seed(42)
print(f'Data: {_DS["data_file"]}  symbol={SYMBOL}')


Data: <repo>/data/nq-mbo-sample-15min.dbn.zst  symbol=NQ


## Section 01 — Historical Cascade Identification (skeleton)

In [ ]:
# Heuristic cascade definition: mid-price drops > 8 ticks in 5s = cascade event.
# (In production, use validated cascade taxonomy from p6.cup_flip; this is a smoketest.)
snaps = await collect_snapshots(_DS)
assert len(snaps) > 100, '01 │ too few snapshots for cascade detection'

mids, ts = [], []
for s in snaps:
    if s.bids and s.asks:
        mids.append((s.bids[0].price + s.asks[0].price) / 2)
        ts.append(s.timestamp_ms)
mids = np.array(mids); ts = np.array(ts)

CASCADE_TICKS = 8
CASCADE_HORIZON_MS = 5_000
cascade_idx = []
for i in range(len(mids)):
    j_end = int(np.searchsorted(ts, ts[i] + CASCADE_HORIZON_MS))
    if j_end <= i: continue
    drop_ticks = (mids[i] - mids[i:j_end].min()) / TICK_SIZE
    if drop_ticks >= CASCADE_TICKS:
        cascade_idx.append(i)
# De-dup: keep one cascade per 30s window
deduped = []
last = -10**9
for i in cascade_idx:
    if ts[i] - last > 30_000:
        deduped.append(i); last = ts[i]
cascade_idx = deduped
log.info('01 │ %d cascade events identified (>=%d-tick drop in %dms)',
         len(cascade_idx), CASCADE_TICKS, CASCADE_HORIZON_MS)


# --- Real taxonomy classifier (Wave 2 #3): run alongside the heuristic ---
# Production thresholds require 60s stalls and 3-length streaks — on 15-min
# smoketest data these never trigger. Use scale-aware thresholds: small
# samples get relaxed cutoffs AND a relaxed TapeReader streak detector
# (min_streak_length=2 instead of 3). Wave 4 Phase 1C.
from p6lab.patterns.cascade_taxonomy import CascadeThresholds
if len(snaps) < 5_000:
    tax_thresholds = CascadeThresholds(
        momentum_min_streak_length=2,
        momentum_min_streak_velocity=1.0,
        liquidity_min_stall_duration_ms=15_000,   # 15s instead of 60s
    )
    tax_min_streak = 2   # relax TapeReader's internal StreakDetector too
    log.info('01 │ using smoketest-scale taxonomy thresholds (n_snaps=%d)', len(snaps))
else:
    tax_thresholds = None   # production defaults
    tax_min_streak = None
    log.info('01 │ using production taxonomy thresholds (n_snaps=%d)', len(snaps))

clf = CascadeClassifier(tick_size=TICK_SIZE, thresholds=tax_thresholds,
                        min_streak_length=tax_min_streak)
tax_events = clf.classify_snapshots(snaps)
tax_df = cascade_events_to_df(tax_events)
log.info('01 │ taxonomy events: %d total  (A=%d B=%d C=%d D=%d)',
         len(tax_events),
         sum(1 for e in tax_events if e.cascade_type == CascadeType.LIQUIDITY_WITHDRAWAL),
         sum(1 for e in tax_events if e.cascade_type == CascadeType.MOMENTUM_IGNITION),
         sum(1 for e in tax_events if e.cascade_type == CascadeType.GRINDING_CORRECTION),
         sum(1 for e in tax_events if e.cascade_type == CascadeType.CROSS_INSTRUMENT_CONTAGION))
if not tax_df.empty:
    log.info('01 │ sample taxonomy events:\n%s', tax_df.head(5).to_string(index=False))

# Union: use both heuristic and taxonomy anchors for downstream analysis
tax_anchors = [e.anchor_ts_ms for e in tax_events]
log.info('01 │ heuristic cascades=%d  taxonomy cascades=%d', len(cascade_idx), len(tax_events))


## Section 02 — Pre-Cascade Window Extraction (skeleton)

In [3]:
# For each cascade, compute FI 30s before. Negative (control) sample = FI at random non-cascade times.
adapter = L1Adapter(L1AdapterConfig(tick_size=TICK_SIZE))
l1_rows = [compute_l1_features(adapter.ingest(s), adapter.history) for s in snaps]
l1_df = pd.DataFrame(np.array(l1_rows), columns=L1FeatureNames.ALL)

l2_hist = L2History()
l2_rows = []
for s in snaps:
    if not (s.bids and s.asks): continue
    bid_map = {lvl.price: lvl.volume for lvl in s.bids[:20]}
    ask_map = {lvl.price: lvl.volume for lvl in s.asks[:20]}
    prices = sorted(set(bid_map) | set(ask_map), reverse=True)
    book_levels = [(p, bid_map.get(p, 0.0), ask_map.get(p, 0.0)) for p in prices]
    l2_snap = L2Snapshot(timestamp_ms=s.timestamp_ms, symbol=SYMBOL,
                         mid_price=(s.bids[0].price + s.asks[0].price)/2,
                         book_levels=book_levels)
    l2_hist.append(l2_snap)
    feat = compute_l2_features(l2_snap, l2_hist)
    l2_rows.append(feat)
l2_arr = np.array(l2_rows)
l1_arr = l1_df.to_numpy()
n = min(len(l1_arr), len(l2_arr))

fi = FragilityIndex()
fi_full = []
for i in range(n):
    sub = fi.compute_sub_indices(l1_arr[i], l2_arr[i], vpin_value=0.0)
    fi_full.append(fi.compute_full(sub))
fi_full = np.array(fi_full)

# Lead-time labels: 1 if any cascade occurs in next 30s, 0 otherwise
LEAD_MS = 30_000
labels_arr = np.zeros(n, dtype=int)
cascade_ts = ts[cascade_idx] if cascade_idx else np.array([])
for i in range(n):
    if cascade_ts.size:
        future = cascade_ts[(cascade_ts > ts[i]) & (cascade_ts <= ts[i] + LEAD_MS)]
        if len(future): labels_arr[i] = 1
log.info('02 │ FI series len=%d, label rate=%.2f%%',
         n, 100*labels_arr.mean())


02 │ FI series len=2000, label rate=▒.▒▒%


## Section 03 — CPCV with 14-day Embargo

In [ ]:
cv = CascadeAwareCPCV(n_splits=5, n_test_groups=2, cascade_embargo_days=14)
# Demo-only local scope: don't shadow the real `n` from §02 (which §05
# relies on). Audit item F root cause.
_n_demo = 200
_X_demo = pd.DataFrame(np.zeros((_n_demo, 4)))
_ts_demo = pd.Series(pd.date_range('2025-01-01', periods=_n_demo, freq='1D'))
_cascades_demo = pd.Series([_ts_demo.iloc[100]])
_folds_demo = cv.split(_X_demo, _ts_demo, _cascades_demo)
log.info('03 │ %d folds, total embargoed rows: %d',
         len(_folds_demo), sum(len(f.embargoed_idx) for f in _folds_demo))


## Section 04 — FI Predictive Power (target AUC > 0.70, skeleton)

In [ ]:
# Skip CPCV at this scale — single in-sample AUC is enough for smoketest.
# Trim to common length before computing AUC (labels_arr and fi_full may
# differ by a few rows if snapshots had empty books). Don't use §03's `n`,
# which is a demo override for the CPCV cell. Audit item F.
n_labels = len(labels_arr)
n_fi = len(fi_full)
if n_labels != n_fi:
    log.warning('04 │ size mismatch: labels=%d fi_full=%d — trimming to common',
                n_labels, n_fi)
    m = min(n_labels, n_fi)
    labels_arr = labels_arr[:m]
    fi_full = fi_full[:m]

n_rows = len(labels_arr)
n_pos = int(labels_arr.sum())
if 0 < n_pos < n_rows:
    fi_auc = float(roc_auc_score(labels_arr, fi_full))
    log.info('04 │ FI AUC for cascade prediction: %.3f (target ≥0.70, %d/%d positives)',
             fi_auc, n_pos, n_rows)
else:
    fi_auc = float('nan')
    log.info('04 │ FI AUC skipped (degenerate: %d/%d positives)', n_pos, n_rows)


## Section 05 — Lead-time Distribution (≥2min before 50% of cascades)

In [6]:
# For each cascade, find earliest tick where FI exceeded 0.7
FI_THRESHOLD = 0.7
lead_times_ms = []
for cidx in cascade_idx:
    if cidx >= n: continue
    # Walk backward from cascade time, find earliest time where FI > threshold
    # within the previous 5 minutes
    look_back = max(0, cidx - 3000)  # ~5 min at 100ms snaps
    flagged = np.where(fi_full[look_back:cidx] > FI_THRESHOLD)[0]
    if len(flagged):
        first_flag = look_back + flagged[0]
        lead_times_ms.append(int(ts[cidx] - ts[first_flag]))
if lead_times_ms:
    arr = np.array(lead_times_ms) / 1000.0
    log.info('05 │ lead time (sec): mean=%.1f median=%.1f n=%d (target: median ≥120s)',
             arr.mean(), np.median(arr), len(arr))
else:
    log.info('05 │ no FI flags ahead of any cascade')


05 │ no FI flags ahead of any cascade


## Section 06 — FP Rate at FI > 0.7 (target <30%)

In [7]:
flag_mask = fi_full > FI_THRESHOLD
if flag_mask.sum() > 0:
    fp_rate = float(((~labels_arr.astype(bool)) & flag_mask).sum() / flag_mask.sum())
    log.info('06 │ FP rate at FI>%.1f: %.1f%% (target <30%%)',
             FI_THRESHOLD, 100*fp_rate)
else:
    fp_rate = float('nan')
    log.info('06 │ no FI flags at threshold %.1f', FI_THRESHOLD)


06 │ no FI flags at threshold ▒.▒▒


## Section 07 — Synthetic Augmentation Comparison

In [8]:
eng = AugmentationEngine(random_state=42)
demo = pd.DataFrame({
    'bid_size': np.linspace(10, 20, 50),
    'ask_size': np.linspace(20, 10, 50),
    'spread_ticks': np.ones(50),
})
samples = eng.generate(demo, label=1, instrument='NQ', n_samples=6)
log.info('07 │ generated %d augmented samples (methods: %s)',
         len(samples), sorted({s.augmentation_method for s in samples}))


07 │ generated 6 augmented samples (methods: ['depth_scale', 'original', 'phase_duration_jitter', 'time_compress', 'time_stretch', 'volatility_shift'])


## Section 08 — Per-Cascade-Type Classifier (Type A/B/C/D, skeleton)

In [9]:
# Per-cascade-type stats — how do Type A vs B behave differently?
if tax_events:
    per_type_rows = []
    for ct in CascadeType:
        members = [e for e in tax_events if e.cascade_type == ct]
        if not members:
            continue
        mean_conf = float(np.mean([e.confidence for e in members]))
        mean_dur_ms = float(np.mean([e.end_ts_ms - e.anchor_ts_ms for e in members]))
        per_type_rows.append({
            'cascade_type': ct.value,
            'n': len(members),
            'mean_confidence': round(mean_conf, 3),
            'mean_duration_ms': round(mean_dur_ms, 1),
        })
    per_type_df = pd.DataFrame(per_type_rows)
    log.info('08 │ per-type cascade stats:\n%s', per_type_df.to_string(index=False))
else:
    per_type_df = pd.DataFrame()
    log.info('08 │ no taxonomy cascades — classifier thresholds may need tuning at this scale')
per_type_df


08 │ no taxonomy cascades — classifier thresholds may need tuning at this scale


""


## Section 09 — Cross-Instrument Stress (CIS, skeleton)

In [10]:
fi = FragilityIndex()
sub = fi.compute_sub_indices(np.zeros(16), np.zeros(12),
                             vpin_value=0.3, cross_instrument_stress=0.5)
log.info('09 │ FI sub-indices: DF=%.2f CF=%.2f RF=%.2f SF=%.2f FT=%.2f CIS=%.2f',
         sub.DF, sub.CF, sub.RF, sub.SF, sub.FT, sub.CIS)


09 │ FI sub-indices: DF=▒.▒▒ CF=▒.▒▒ RF=▒.▒▒ SF=▒.▒▒ FT=▒.▒▒ CIS=▒.▒▒


## Section 10 — Report + FI Model Export (skeleton)

Persist FI parameters for the web server fragility gauge (§10.5, §11.4).

In [11]:
out_dir = versioned_dir(ARTIFACTS_DIR / 'cascade', 'nb07', data_slice=_DS,
                        extra={'n_cascades_heuristic': len(cascade_idx),
                               'n_taxonomy_cascades': len(tax_events),
                               'taxonomy_by_type': {ct.value: sum(1 for e in tax_events if e.cascade_type == ct) for ct in CascadeType},
                               'fi_auc': fi_auc if not np.isnan(fi_auc) else None,
                               'fp_rate_at_0.7': fp_rate if not np.isnan(fp_rate) else None})
report_path = out_dir / 'nb07_report.md'
with open(report_path, 'w') as f:
    f.write(f'# NB07 — Cascade Lab Report\n\n')
    f.write(f'**Symbol:** {SYMBOL}\n')
    f.write(f'**Snapshots:** {len(snaps)}\n')
    f.write(f'**Cascades identified:** {len(cascade_idx)}\n')
    f.write(f'**FI AUC (cascade prediction):** {fi_auc:.3f}\n')
    f.write(f'**FP rate at FI>0.7:** {fp_rate:.1%}\n')
log.info('10 │ wrote %s', report_path)


10 │ wrote <repo>/p6lab/artifacts/p6lab/cascade/nb07_20260420_1648/nb07_report.md
